In [119]:
import os
import pickle
import random

classes = ['1', '2', '3', '4', '5', '+', '-', '=', 'del']

test_set = []
train_set = []

for i in range(9):
    with open(os.path.join('data/raw', f'{classes[i]}.pkl'), 'rb') as f:
        arr = pickle.load(f)
    for j in range(90):
        a = []
        for b in arr[j]:
            a.append(b[0])
            a.append(b[1])
            a.append(b[2])
        train_set.append([a, i])
    for j in range(90, 100):
        a = []
        for b in arr[j]:
            a.append(b[0])
            a.append(b[1])
            a.append(b[2])
        test_set.append([a, i])
        
random.shuffle(test_set)
random.shuffle(train_set)
    

In [120]:
import torch

class MLP_hands(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.W1 = torch.nn.Parameter(torch.randn(63, 128) * 0.1)
        self.W2 = torch.nn.Parameter(torch.randn(128, 64) * 0.1)
        self.W3 = torch.nn.Parameter(torch.randn(64, 9) * 0.1)
        self.B1 = torch.nn.Parameter(torch.zeros(128))
        self.B2 = torch.nn.Parameter(torch.zeros(64))
        self.B3 = torch.nn.Parameter(torch.zeros(9))
    
    def forward(self, a1):
        z1 = a1 @ self.W1 + self.B1
        a2 = torch.relu(z1)
        z2 = a2 @ self.W2 + self.B2
        a3 = torch.relu(z2)
        ret = a3 @ self.W3 + self.B3
        return ret
    

In [121]:
model = MLP_hands()

batch_size = 3
ch = 0.01
f = torch.nn.CrossEntropyLoss()

for e in range(100):
    random.shuffle(train_set)
    for _ in range(0, len(train_set), batch_size):
        batch = torch.tensor([x[0] for x in train_set[_: _ + batch_size]], dtype=torch.float32)
        ans = torch.tensor([x[1] for x in train_set[_: _ + batch_size]], dtype=torch.long)
        ret = model.forward(batch)
        loss = f(ret, ans)
        loss.backward()
        with torch.no_grad():
            for smth in [model.W1, model.W2, model.W3, model.B1, model.B2, model.B3]:
                smth.sub_(ch * smth.grad)
                smth.grad = None
    print(e)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [122]:
cnt = 0
for x in test_set:
    batch = torch.tensor([x[0]], dtype=torch.float32)
    ret = model.forward(batch)
    imax = 0
    for i in range(9):
        if ret[0][i] > ret[0][imax]:
            imax = i
    ret = ret.softmax(dim = 1)
    print(ret[0][imax])
    if imax == x[1]:
        cnt += 1
print(cnt / len(test_set))

tensor(0.9964, grad_fn=<SelectBackward0>)
tensor(0.9981, grad_fn=<SelectBackward0>)
tensor(0.9985, grad_fn=<SelectBackward0>)
tensor(0.9909, grad_fn=<SelectBackward0>)
tensor(0.9986, grad_fn=<SelectBackward0>)
tensor(0.9506, grad_fn=<SelectBackward0>)
tensor(0.9901, grad_fn=<SelectBackward0>)
tensor(0.9982, grad_fn=<SelectBackward0>)
tensor(0.9961, grad_fn=<SelectBackward0>)
tensor(0.9985, grad_fn=<SelectBackward0>)
tensor(0.9986, grad_fn=<SelectBackward0>)
tensor(0.9910, grad_fn=<SelectBackward0>)
tensor(0.9991, grad_fn=<SelectBackward0>)
tensor(0.9112, grad_fn=<SelectBackward0>)
tensor(0.9990, grad_fn=<SelectBackward0>)
tensor(0.9481, grad_fn=<SelectBackward0>)
tensor(0.9949, grad_fn=<SelectBackward0>)
tensor(0.9890, grad_fn=<SelectBackward0>)
tensor(0.9998, grad_fn=<SelectBackward0>)
tensor(0.9991, grad_fn=<SelectBackward0>)
tensor(0.9992, grad_fn=<SelectBackward0>)
tensor(0.9974, grad_fn=<SelectBackward0>)
tensor(0.9983, grad_fn=<SelectBackward0>)
tensor(0.9990, grad_fn=<SelectBack

In [123]:
torch.save(model.state_dict(), 'working_model.pth')